In [5]:
import pandas as pd
import ast

In [33]:
df = pd.read_json('GymShockGif.json')
df.head()

,bodyPart,equipment,id,target,secondaryMuscles,instructions,name,gifUrl
0,waist,body weight,1,abs,"[hip flexors, lower back]",[Lie flat on your back with your knees bent an...,3/4 sit-up,https://gymvisual.com/img/p/4/7/3/1/4731.gif
1,waist,body weight,3,abs,[hip flexors],[Lie flat on your back with your hands placed ...,air bike,https://gymvisual.com/img/p/8/8/0/8/8808.gif
2,upper legs,body weight,1512,quads,"[hamstrings, glutes]",[Start on all fours with your hands directly u...,all fours squad stretch,https://gymvisual.com/img/p/6/9/7/9/6979.gif
3,waist,body weight,6,abs,[obliques],[Lie flat on your back with your knees bent an...,alternate heel touchers,https://gymvisual.com/img/p/1/7/3/2/0/17320.gif
4,back,cable,7,lats,"[biceps, rhomboids]",[Sit on the cable machine with your back strai...,alternate lateral pulldown,https://gymvisual.com/img/p/4/7/3/8/4738.gif


In [43]:
exercises = df[['bodyPart', 'equipment', 'target', 'secondaryMuscles', 'instructions', 'name']].copy()

In [44]:
exercises['equipment']=exercises['equipment'].apply(lambda x: x.split())
exercises['bodyPart']=exercises['bodyPart'].apply(lambda x: x.split())
exercises['target']=exercises['target'].apply(lambda x: x.split())

In [45]:
exercises['secondaryMuscles']=exercises['secondaryMuscles'].apply(lambda x: [i.replace(' ','') for i in x])
#exercises['instructions']=exercises['instructions'].apply(lambda x: [i.replace(' ',', ') for i in x])

In [46]:
exercises['tags']=exercises['bodyPart'] + exercises['equipment'] + exercises['target'] + exercises['secondaryMuscles'] + exercises['instructions']

In [47]:
exercises['tags']=exercises['tags'].apply(lambda x: ' '.join(x))

In [48]:
exercises.drop(['bodyPart', 'equipment', 'target', 'secondaryMuscles', 'instructions'], axis=True, inplace=True)

In [49]:
exercises.head()

,name,tags
0,3/4 sit-up,waist body weight abs hipflexors lowerback Lie...
1,air bike,waist body weight abs hipflexors Lie flat on y...
2,all fours squad stretch,upper legs body weight quads hamstrings glutes...
3,alternate heel touchers,waist body weight abs obliques Lie flat on you...
4,alternate lateral pulldown,back cable lats biceps rhomboids Sit on the ca...


In [50]:
from sklearn.feature_extraction.text import CountVectorizer

In [51]:
cv = CountVectorizer(max_features=5000, stop_words='english')

In [57]:
vector = cv.fit_transform(exercises['tags']).toarray()
vector.shape

(1094, 779)

In [58]:
from sklearn.metrics.pairwise import cosine_similarity

In [60]:
similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.38854367, 0.29508445, ..., 0.27971546, 0.29101262,
        0.21821789],
       [0.38854367, 1.        , 0.22357373, ..., 0.20064309, 0.49932203,
        0.18314031],
       [0.29508445, 0.22357373, 1.        , ..., 0.14285714, 0.39931084,
        0.19317812],
       ...,
       [0.27971546, 0.20064309, 0.14285714, ..., 1.        , 0.10519479,
        0.25354628],
       [0.29101262, 0.49932203, 0.39931084, ..., 0.10519479, 1.        ,
        0.12192799],
       [0.21821789, 0.18314031, 0.19317812, ..., 0.25354628, 0.12192799,
        1.        ]], shape=(1094, 1094))

In [70]:
index = exercises[exercises['name']=='3/4 sit-up'].index[0]
print(similarity[index])
print(enumerate(similarity[index]))
print(list(enumerate(similarity[index])))
print(sorted(list(enumerate(similarity[index])), reverse=True))
print(sorted(list(enumerate(similarity[index])), reverse=True, key=lambda x: x[1]))
print(sorted(list(enumerate(similarity[index])), reverse=True, key=lambda x: x[1])[1:6])
distance = sorted(list(enumerate(similarity[index])), reverse=True, key=lambda x: x[1])[1:6]
print([exercises.iloc[x[0]]['name'] for x in distance])

[1.         0.38854367 0.29508445 ... 0.27971546 0.29101262 0.21821789]
[(0, np.float64(0.9999999999999998)), (1, np.float64(0.3885436701001307)), (2, np.float64(0.29508444542532697)), (3, np.float64(0.4853238255909908)), (4, np.float64(0.26599758829946574)), (5, np.float64(0.287824837800531)), (6, np.float64(0.49488120534872543)), (7, np.float64(0.4303314829119351)), (8, np.float64(0.24688535993934702)), (9, np.float64(0.8720815992723808)), (10, np.float64(0.334687173157217)), (11, np.float64(0.40824829046386296)), (12, np.float64(0.2556038601690775)), (13, np.float64(0.11547005383792515)), (14, np.float64(0.4228900316110309)), (15, np.float64(0.3402069087198858)), (16, np.float64(0.41522739926869984)), (17, np.float64(0.2986595567936802)), (18, np.float64(0.11476952126219177)), (19, np.float64(0.2810013320855967)), (20, np.float64(0.22597307314641285)), (21, np.float64(0.9256514468702007)), (22, np.float64(0.34156502553198653)), (23, np.float64(0.22592402852876595)), (24, np.float64(

In [71]:
def recommend(exercise):
    index = exercises[exercises['name']==exercise].index[0]
    distance = sorted(list(enumerate(similarity[index])), reverse=True, key=lambda x: x[1])[1:6]
    print([exercises.iloc[x[0]]['name'] for x in distance])

In [73]:
recommend('air bike')

['lying elbow to knee', 'band bicycle crunch', 'spine twist', 'knee touch crunch', 'cross body crunch']


In [75]:
df.head()

,bodyPart,equipment,id,target,secondaryMuscles,instructions,name,gifUrl
0,waist,body weight,1,abs,"[hip flexors, lower back]",[Lie flat on your back with your knees bent an...,3/4 sit-up,https://gymvisual.com/img/p/4/7/3/1/4731.gif
1,waist,body weight,3,abs,[hip flexors],[Lie flat on your back with your hands placed ...,air bike,https://gymvisual.com/img/p/8/8/0/8/8808.gif
2,upper legs,body weight,1512,quads,"[hamstrings, glutes]",[Start on all fours with your hands directly u...,all fours squad stretch,https://gymvisual.com/img/p/6/9/7/9/6979.gif
3,waist,body weight,6,abs,[obliques],[Lie flat on your back with your knees bent an...,alternate heel touchers,https://gymvisual.com/img/p/1/7/3/2/0/17320.gif
4,back,cable,7,lats,"[biceps, rhomboids]",[Sit on the cable machine with your back strai...,alternate lateral pulldown,https://gymvisual.com/img/p/4/7/3/8/4738.gif


In [76]:
df[df['name']=='lying elbow to knee']

,bodyPart,equipment,id,target,secondaryMuscles,instructions,name,gifUrl
846,waist,body weight,2312,abs,"[hip flexors, lower back]",[Lie flat on your back with your knees bent an...,lying elbow to knee,https://gymvisual.com/img/p/9/7/9/7/9797.gif


In [74]:
import pickle

In [77]:
pickle.dump(exercises, open('exercises.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
exercises = pickle.load(open('exercises.pkl', 'rb'))

In [78]:
exercises = pickle.load(open('exercises.pkl', 'rb'))
exercises

,name,tags
0,3/4 sit-up,waist body weight abs hipflexors lowerback Lie...
1,air bike,waist body weight abs hipflexors Lie flat on y...
2,all fours squad stretch,upper legs body weight quads hamstrings glutes...
3,alternate heel touchers,waist body weight abs obliques Lie flat on you...
4,alternate lateral pulldown,back cable lats biceps rhomboids Sit on the ca...
...,...,...
1089,wide grip rear pull-up,back body weight lats biceps forearms Grab the...
1090,wide hand push up,chest body weight pectorals triceps shoulders ...
1091,wind sprints,waist body weight abs quadriceps hamstrings ca...
1092,world greatest stretch,upper legs body weight hamstrings glutes quadr...
